In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn import svm

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
# data = pd.read_excel('dataset_all_original.xlsx') 
# data = pd.read_excel('dataset_all_gen_ssdp.xlsx') 
data = pd.read_excel('dataset_all_gen.xlsx') 
# data = pd.read_excel('dataset_all_original_ssdp.xlsx') 
# data = pd.read_excel('dataset_all_original_add_nobias.xlsx') 

In [4]:
data = data.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
Name             0
Vendor           0
DNS           2473
mDNS          3267
NUPNP        11699
open_port     1916
SSDP_Type     2688
SSDP_t       11384
SSDP_v       11384
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
data.isnull().sum()

type         0
Name         0
Vendor       0
DNS          0
mDNS         0
NUPNP        0
open_port    0
SSDP_Type    0
SSDP_t       0
SSDP_v       0
dtype: int64

In [9]:
data.dtypes

type         object
Name         object
Vendor       object
DNS          object
mDNS         object
NUPNP        object
open_port    object
SSDP_Type    object
SSDP_t       object
SSDP_v       object
dtype: object

In [10]:
data = data.astype('category')

In [11]:
data.dtypes

type         category
Name         category
Vendor       category
DNS          category
mDNS         category
NUPNP        category
open_port    category
SSDP_Type    category
SSDP_t       category
SSDP_v       category
dtype: object

In [12]:
data

,type,Name,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Energy Management,TP-Link Smart WiFi LED Bulb,"TP-LINK TECHNOLOGIES CO.,LTD.",booking.com,NaN,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
1,Health-Monitor,Withings Smart Monitor,Withings,discuz.net,_stronghold._udp.local.,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
2,Appliances,Bezeq smarthome,Bezeq Group,soundcloud.com,NaN,NaN,tcp/80,NOTIFY * HTTP/1.1,NaN,NaN
3,Energy Management,TP-Link Smart WiFi LED Bulb,"TP-LINK TECHNOLOGIES CO.,LTD.",auda.org.au,_zaam-dox._udp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
4,Appliances,Bose SoundTouch 10,Bose Corporation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Energy Management,Belkin WeMo Link,Belkin International Inc.,artisteer.com,_alphazap._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NaN,NaN
6,Hubs,Philips HUE Hub,Philips Lighting BV,dagondesign.com,_wrapsafe._udp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN
7,Hubs,2X Amazone Echo,Amazon Technologies Inc.,telegraph.co.uk,_prodder._tcp.local.,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
8,Non-IOT,LG G3 Smartphone,LG Electronics,jimdo.com,_prodder._udp.local.,NaN,tcp/443,M-SEARCH * HTTP/1.1,NaN,NaN
9,Health-Monitor,Withings Smart scale,Withings,oaic.gov.au,NaN,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN


In [13]:
y = data.drop_duplicates(keep='first')
# y = y.drop(['name'], axis=1)
# y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [14]:
y = y.astype(str)
len(y)

11469

In [15]:
y = y.apply(LabelEncoder().fit_transform)

In [16]:
list_data = []
list_data.append(y)
len(list_data)

1

In [17]:
accuracy = []
accuracy_t = []
acc = []

In [18]:
for data in list_data:
    for i in range(1):
        accuracy_temp = []
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            clf = svm.SVC(kernel='linear', decision_function_shape='ovr')
            clf = clf.fit(train_data,label_data )
            predict = clf.predict(test_data)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            acc.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  0.46033129904097647
0   Accuracy:  0.4734088927637315
0   Accuracy:  0.44376634699215345
0   Accuracy:  0.4899738448125545
0   Accuracy:  0.4812554489973845
0   Accuracy:  0.4908456843940715
0   Accuracy:  0.45510026155187444
0   Accuracy:  0.47602441150828245
0   Accuracy:  0.47602441150828245
0   Accuracy:  0.4825479930191972


In [19]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
# print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
# print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
# print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
# print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
# print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
# print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
# print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
# print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

11469 	 Accuracy for dataset_gen: 	 0.47292785945885085


In [20]:
print(len(y),"\t STD for dataset_gen: \t", np.std(acc))

11469 	 STD for dataset_gen: 	 0.014559563308693402
